# DCOPF - Example 2 (Infinite Network, B-theta)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Same B-theta DCOPF as Example 1 but with the line thermal limits removed ("infinite network"). All nodal balances and B-theta flow equations remain. Cheap unit can now dispatch up to its full capacity since no line is binding.


In [1]:
from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint, SolverFactory,
    minimize, value
)

c1, Pgmin1, Pgmax1 = 10, 20, 70
c3, Pgmin3, Pgmax3 = 20, 40, 90
Load2 = 100
x = 0.1
BaseMW = 100

model = ConcreteModel()
model.G1 = Var(bounds=(Pgmin1, Pgmax1))
model.G3 = Var(bounds=(Pgmin3, Pgmax3))
model.pk1 = Var()
model.pk2 = Var()
model.pk3 = Var()
model.theta1 = Var()
model.theta2 = Var()
model.theta3 = Var()

model.obj = Objective(expr=c1*model.G1 + c3*model.G3, sense=minimize)

model.lineFlow_1 = Constraint(expr=model.pk1/BaseMW == (model.theta1 - model.theta2)/x)
model.lineFlow_2 = Constraint(expr=model.pk2/BaseMW == (model.theta1 - model.theta3)/x)
model.lineFlow_3 = Constraint(expr=model.pk3/BaseMW == (model.theta2 - model.theta3)/(2*x))

model.PowerBalance_1 = Constraint(expr=model.G1 - model.pk1 - model.pk2 == 0)
model.PowerBalance_2 = Constraint(expr=model.pk1 - model.pk3 == Load2)
model.PowerBalance_3 = Constraint(expr=model.G3 + model.pk2 + model.pk3 == 0)

model.theta3.fix(0)

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
print(f"G1 = {value(model.G1):.4f}, G3 = {value(model.G3):.4f}")
print(f"pk1 = {value(model.pk1):.4f}, pk2 = {value(model.pk2):.4f}, pk3 = {value(model.pk3):.4f}")
print(f"theta1 = {value(model.theta1):.4f}, theta2 = {value(model.theta2):.4f}, theta3 = {value(model.theta3):.4f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpblrg201c.pyomo.lp


Reading time = 0.00 seconds
x1: 6 rows, 7 columns, 15 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90


MIPGap  0

Optimize a model with 6 rows, 7 columns and 15 nonzeros
Model fingerprint: 0x9bac4631
Coefficient statistics:
  Matrix range     [1e-02, 1e+01]


  Objective range  [1e+01, 2e+01]
  Bounds range     [2e+01, 9e+01]
  RHS range        [1e+02, 1e+02]
Presolve removed 6 rows and 7 columns


Presolve time: 0.00s
Presolve: All rows and columns removed


Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.4000000e+03   0.000000e+00   0.000000e+00      0s



Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.400000000e+03


ok optimal
G1 = 60.0000, G3 = 40.0000
pk1 = 65.0000, pk2 = -5.0000, pk3 = -35.0000
theta1 = -0.0050, theta2 = -0.0700, theta3 = 0.0000
